# VisionBridge — Automatic Real-Video Validation

Run every cell from top to bottom. This notebook automatically downloads the real ISL-CSLTR dataset, selects one sentence-level video deterministically, derives its ground-truth sentence from the dataset folder label, extracts real MediaPipe Holistic pose/face features, loads the committed VisionBridge base model, and prints the prediction report.

This notebook is validation-only. It does not train or modify the model.

In [ ]:
from pathlib import Path
import os, sys, subprocess
REPO = Path('/content/VisionBridge')
if not (REPO / 'README.md').exists():
    subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)
if str(REPO/'backend') not in sys.path:
    sys.path.insert(0, str(REPO/'backend'))
os.chdir(REPO)
print('Repository:', REPO)
print('HEAD:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'], text=True).strip())
print('REPO SYNC: PASS')

## Step 2 — Runtime check

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU: CPU fallback')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Step 3 — Download the real ISL-CSLTR dataset and automatically select one sentence video

In [ ]:
import glob, random, subprocess, sys
try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','kagglehub'], check=True)
    import kagglehub

DATASET_PATH = kagglehub.dataset_download('drblack00/isl-csltr-indian-sign-language-dataset')
candidates = [p for p in glob.glob(os.path.join(DATASET_PATH,'**','*Sentence_Level*'), recursive=True) if os.path.isdir(p) and 'Video' in os.path.basename(p)]
if len(candidates) != 1:
    raise RuntimeError(f'Expected exactly one sentence-level video directory, found: {candidates}')
VIDEO_ROOT = candidates[0]
video_files = []
for ext in ('*.mp4','*.MP4','*.avi','*.AVI','*.mov','*.MOV','*.mkv','*.MKV'):
    video_files.extend(glob.glob(os.path.join(VIDEO_ROOT,'**',ext), recursive=True))
video_files = sorted(video_files)
if not video_files:
    raise RuntimeError(f'No sentence-level videos found under {VIDEO_ROOT}')

SEED = 42
rng = random.Random(SEED)
VIDEO_PATH = Path(rng.choice(video_files))
GROUND_TRUTH = VIDEO_PATH.parent.name.replace('_',' ').strip()

print('Dataset:', DATASET_PATH)
print('Sentence-video root:', VIDEO_ROOT)
print('Available sentence videos:', len(video_files))
print('Selected video:', VIDEO_PATH)
print('Ground truth from dataset label:', repr(GROUND_TRUTH))
if not GROUND_TRUTH:
    raise RuntimeError('Selected video has no usable label from its parent directory.')

## Step 4 — Prepare the isolated MediaPipe environment

In [ ]:
import shutil, textwrap
UV = shutil.which('uv') or '/usr/local/bin/uv'
if not Path(UV).exists():
    subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
    UV = '/root/.local/bin/uv'
MP_ENV = Path('/content/visionbridge_mp312')
MP_PYTHON = MP_ENV/'bin'/'python'
MPL_CONFIG = Path('/content/visionbridge_mplconfig')
MPL_CONFIG.mkdir(parents=True, exist_ok=True)
if not MP_ENV.exists():
    subprocess.run([UV,'python','install','3.12'], check=True)
    subprocess.run([UV,'venv','--python','3.12',str(MP_ENV)], check=True)
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
env['MPLCONFIGDIR'] = str(MPL_CONFIG)
probe = subprocess.run([str(MP_PYTHON),'-c','import mediapipe; from mediapipe.python.solutions import holistic; print(mediapipe.__version__)'], text=True, capture_output=True, env=env)
if probe.returncode != 0 or probe.stdout.strip() != '0.10.21':
    subprocess.run([UV,'pip','install','--python',str(MP_PYTHON),'mediapipe==0.10.21','numpy==1.26.4','opencv-python-headless','pandas','matplotlib'], check=True, env=env)
probe = subprocess.run([str(MP_PYTHON),'-c','import mediapipe; from mediapipe.python.solutions import holistic; print(mediapipe.__version__)'], text=True, capture_output=True, env=env)
print(probe.stdout.strip())
if probe.returncode != 0:
    print(probe.stderr)
    raise RuntimeError('MediaPipe environment validation failed.')
print('MEDIAPIPE ENV: PASS')

## Step 5 — Extract real pose and face features from the selected video

In [ ]:
import csv, numpy as np, shutil
CHECK_DIR = REPO/'data'/'model_check'
VIDEO_DIR = CHECK_DIR/'videos'
PROCESSED = CHECK_DIR/'processed'
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
UID = VIDEO_PATH.stem
VIDEO_COPY = VIDEO_DIR/VIDEO_PATH.name
shutil.copy2(VIDEO_PATH, VIDEO_COPY)
LABELS = CHECK_DIR/'validation_labels.csv'
with LABELS.open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f); writer.writerow(['uid','text']); writer.writerow([UID, GROUND_TRUTH])
cmd = [str(MP_PYTHON), str(REPO/'backend'/'scripts'/'extract_keypoints.py'), '--videos_dir', str(VIDEO_DIR), '--labels_csv', str(LABELS), '--out_dir', str(PROCESSED)]
result = subprocess.run(cmd, cwd=str(REPO), env=env, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Real MediaPipe extraction failed.')
pose = np.load(PROCESSED/'pose'/f'{UID}.npy')
face = np.load(PROCESSED/'face'/f'{UID}.npy')
print('Pose shape:', pose.shape)
print('Face shape:', face.shape)
if pose.ndim != 2 or pose.shape[1] != 132: raise ValueError(f'Invalid pose shape: {pose.shape}')
if face.ndim != 2 or face.shape[1] != 1404: raise ValueError(f'Invalid face shape: {face.shape}')
if pose.shape[0] != face.shape[0]: raise ValueError('Pose/face frame counts differ.')
print('REAL KEYPOINT EXTRACTION: PASS')

## Step 6 — Run the committed model and print Ground Truth vs Predicted

In [ ]:
import torch
from app.models.base_model import load_frozen_base_model
from app.training.isltranslate import SimpleCharTokenizer, _downsample_to_max_length
WEIGHTS = REPO/'backend/app/models/weights/base_model.pt'
VOCAB = REPO/'backend/app/models/weights/base_model.vocab.json'
if not WEIGHTS.exists(): raise FileNotFoundError(WEIGHTS)
if not VOCAB.exists(): raise FileNotFoundError(VOCAB)
tokenizer = SimpleCharTokenizer.load(VOCAB)
model = load_frozen_base_model(str(WEIGHTS), vocab_size=tokenizer.vocab_size).to(DEVICE).eval()
pose_t, face_t = _downsample_to_max_length(torch.from_numpy(pose).float(), torch.from_numpy(face).float(), UID)
pose_t = pose_t.unsqueeze(0).to(DEVICE)
face_t = face_t.unsqueeze(0).to(DEVICE)
lengths = torch.tensor([pose_t.shape[1]], dtype=torch.long, device=DEVICE)
with torch.no_grad():
    logits = model(pose_t, face_t, lengths=lengths)
print('Checkpoint:', WEIGHTS)
print('Device:', DEVICE)
print('Logits shape:', tuple(logits.shape))
print('Logits finite:', bool(torch.isfinite(logits).all()))

In [ ]:
# Deterministic greedy CTC decode and metrics.
logit = logits[0, :int(lengths[0].item())]
probs = torch.softmax(logit, dim=-1)
frame_ids = probs.argmax(dim=-1)
frame_conf = probs.max(dim=-1).values
collapsed = []
previous = None
for token_id in frame_ids.tolist():
    if token_id != previous: collapsed.append(token_id)
    previous = token_id
decoded_ids = [i for i in collapsed if i != 0]
prediction = ''.join(tokenizer.id_to_token[i] for i in decoded_ids if 0 <= i < len(tokenizer.id_to_token) and tokenizer.id_to_token[i] != tokenizer.blank_token).strip() or '(no sign detected)'
blank_ratio = float((frame_ids == 0).float().mean().item())
non_blank = int((frame_ids != 0).sum().item())
confidence = float(frame_conf.mean().item())

def levenshtein(a, b):
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0]*len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca != cb))
        prev = cur
    return prev[-1]
cer = levenshtein(prediction.lower(), GROUND_TRUTH.lower()) / max(1, len(GROUND_TRUTH))
print('\n' + '='*72)
print('VISIONBRIDGE AUTOMATIC REAL-VIDEO MODEL VALIDATION')
print('='*72)
print('DATASET:         ISL-CSLTR')
print('SELECTED VIDEO:  ', VIDEO_PATH.name)
print('GROUND TRUTH:    ', GROUND_TRUTH)
print('PREDICTED:       ', prediction)
print('CONFIDENCE:      ', f'{confidence:.3f}')
print('BLANK RATIO:     ', f'{blank_ratio:.4f}')
print('NON-BLANK FRAMES:', non_blank)
print('UNIQUE TOKENS:   ', len(set(decoded_ids)))
print('FRAMES:          ', frame_ids.numel())
print('CER:             ', f'{cer:.4f}')
print('LOGITS FINITE:   ', bool(torch.isfinite(logits).all()))
print('='*72)
if prediction == '(no sign detected)':
    print('RESULT: EMPTY CTC OUTPUT ON THE SELECTED REAL VIDEO.')
else:
    print('RESULT: NON-EMPTY PREDICTION PRODUCED.')

In [ ]:
from IPython.display import Video, display
display(Video(str(VIDEO_COPY), embed=True))